In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import AgglomerativeClustering
from sklearn.metrics import silhouette_score
from scipy.cluster.hierarchy import dendrogram, linkage

In [ ]:
CLEANED_DATA_PATH = "cleaned_marketing_campaign.csv"

df = pd.read_csv(CLEANED_DATA_PATH, parse_dates=["Dt_Customer"])
df.head()

## Feature prep

In [ ]:
df["Total_Spending"] = df[[
    "MntWines", "MntFruits", "MntMeatProducts",
    "MntFishProducts", "MntSweetProducts", "MntGoldProds",
]].sum(axis=1)

df["Total_Purchases"] = df[[
    "NumDealsPurchases", "NumWebPurchases",
    "NumCatalogPurchases", "NumStorePurchases",
]].sum(axis=1)

features = ["Income", "Age", "Total_Spending", "Total_Purchases", "Recency", "Total_Children"]
X = df[features]

In [ ]:
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

In [ ]:
sil_scores = {}
for k in range(2, 10):
    hc = AgglomerativeClustering(n_clusters=k, linkage="ward")
    labels = hc.fit_predict(X_scaled)
    sil_scores[k] = silhouette_score(X_scaled, labels)

print("Silhouette scores by k:", {k: round(v, 3) for k, v in sil_scores.items()})
best_k = max(sil_scores, key=sil_scores.get)
print(f"Best k: {best_k}")

In [ ]:
hc_final = AgglomerativeClustering(n_clusters=best_k, linkage="ward")
df["HC_Cluster"] = hc_final.fit_predict(X_scaled)

print(f"Final silhouette score: {silhouette_score(X_scaled, df['HC_Cluster']):.3f}")
print(df["HC_Cluster"].value_counts().sort_index())

In [ ]:
linkage_matrix = linkage(X_scaled, method="ward")